### Setup

In [ ]:
!pip install rdkit

# From SMILES To A Screening Report

Today you take a small compound library, given only as names and SMILES strings, and carry it all the way to a decision.
You will parse every structure with [RDKit](https://www.rdkit.org/docs/), look at the ones you cannot trust, count atoms and bonds, put everything into a [pandas](https://pandas.pydata.org/docs/) table, filter the table on a drug-likeness rule, compare two chemical classes with a [t-test](https://en.wikipedia.org/wiki/Student%27s_t-test), and finally fit a dissolution curve with [SciPy](https://docs.scipy.org/doc/scipy/).

## Part 1 - Parsing a supplier library

A supplier has sent you eleven compounds as name plus [SMILES](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System) text.
SMILES is only a string, so nothing in it is checked until you try to turn it into a molecule.
That is what `Chem.MolFromSmiles` does: it works out atoms, bonds, valences and aromaticity, and hands back a `Mol` object — or `None` if the string does not describe a real structure.

| name | class | SMILES |
|---|---|---|
| aspirin | NSAID | `CC(=O)Oc1ccccc1C(=O)O` |
| ibuprofen | NSAID | `CC(C)Cc1ccc(cc1)C(C)C(=O)O` |
| naproxen | NSAID | `COc1ccc2cc(ccc2c1)C(C)C(=O)O` |
| salicylic acid | NSAID | `OC(=O)c1ccccc1O` |
| celecoxib | NSAID | `Cc1ccc(cc1)c1cc(nn1c1ccc(cc1)S(N)(=O)=O)C(F)(F)F` |
| caffeine | stimulant | `Cn1cnc2c1c(=O)n(C)c(=O)n2C` |
| theophylline | stimulant | `Cn1c(=O)c2[nH]cnc2n(C)c1=O` |
| theobromine | stimulant | `Cn1cnc2c1c(=O)[nH]c(=O)n2C` |
| nicotine | stimulant | `CN1CCCC1c1cccnc1` |
| paracetamol | analgesic | `CC(=O)Nc1ccc(O)cc1` |
| **compound-X** | unknown | `Cc1ccc(cc` |

The last row is what a real catalogue looks like after somebody edited a file by hand.


#### Questions
  1. Why could the data in the last row of the SMILES column in the above table be a problem?

### Imports

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
from rdkit.Chem.Draw import IPythonConsole

### Looking at the Raw Data
The data from the above table is presented in the [list](https://docs.python.org/3/library/stdtypes.html#typesseq-list)s below.
From the last exercise, you can probably remember that we used [dictionaries](https://docs.python.org/3/tutorial/datastructures.html#dictionaries) - and we will again.
But first, we prepare the data:

In [ ]:
names = ["aspirin", "ibuprofen", "naproxen", "salicylic acid", "celecoxib",
         "caffeine", "theophylline", "theobromine", "nicotine", "paracetamol", "compound-X"]
smiles_list = ["CC(=O)Oc1ccccc1C(=O)O", "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
               "OC(=O)c1ccccc1O", "Cc1ccc(cc1)c1cc(nn1c1ccc(cc1)S(N)(=O)=O)C(F)(F)F",
               "Cn1cnc2c1c(=O)n(C)c(=O)n2C", "Cn1c(=O)c2[nH]cnc2n(C)c1=O", "Cn1cnc2c1c(=O)[nH]c(=O)n2C",
               "CN1CCCC1c1cccnc1", "CC(=O)Nc1ccc(O)cc1", "Cc1ccc(cc"]
classes = ["NSAID", "NSAID", "NSAID", "NSAID", "NSAID",
           "stimulant", "stimulant", "stimulant", "stimulant", "analgesic", "unknown"]
print("entries in the library:", len(names))

#### Questions

2. Parse the **first** SMILES entry in `smiles_list` into an RDKit `Mol` object called `first_mol`.
3. Print the molecule object (`first_mol`) and comment on the output.
4. try to only enter `first_mol` and execute the code block - what happens?


In [ ]:
first_mol = Chem.MolFromSmiles(smiles_list[_])
print(_)
_

One molecule is fine, but the library has 11 entries and one of them is broken.

RDKit does not raise an exception when you give a bad SMILES string.
It returns `None` and prints a warning to the console.


#### Questions
  5. Parse the **last** entry of the `smiles_list` list and convert it to an RDKit molecule. What happens?

In [ ]:
mol = Chem.MolFromSmiles(_)

### Cleaning the Data

To clean the data we will make three new lists: `good_names`, `good_classes` and `good_mols` that only has the data we want to keep.

For good measure, we also keep a list of names for the molecules that failed: `bad_names`.

Lines 1 - 4 below are just definitions of our new lists.
However, line 6 is a monster using the `zip` enumeration that you saw in [video 1.17](https://youtu.be/Of8MPuVeiMw).
#### Questions
  6. Explain in your own words what line 6 (`for name, smiles, cls in zip(names, smiles_list, classes):`) does.
  7. Complete the missing code below (indicated by `_`) so that if a molecule is succesfully parsed, all data goes to the `good_` lists. If a failure happens, the name is recorded in the `bad_names` list.
  8. How many molecules survived, and is that the number you expected?


In [ ]:
good_names = []
good_classes = []
good_mols = []
bad_names = []

for name, smiles, molecule_class in zip(names, smiles_list, classes):
    mol = Chem.MolFromSmiles(_)
    if _ is None:
        bad_names.append(_)
    else:
        good_names.append(_)
        good_classes.append(_)
        good_mols.append(_)

print("succesfully parsed:", len(good_mols), "failed:", bad_names)

The loop used `zip`, which is new: it walks several lists side by side and hands you one item from each per turn, so `name`, `smiles` and `cls` always belong to the same compound.
You will use it again whenever two columns must be read together.

Now that we hold real `Mol` objects, we can ask them chemical questions that the SMILES string could never answer.


#### Questions

9. Caffeine is $\mathrm{C_8H_{10}N_4O_2}$, so it has 24 atoms in total. Predict what `caffeine.GetNumAtoms()` prints before you run the block, and say why.
10. Complete the two marked lines so the block also reports the counts with hydrogens made explicit.
11. Why is the *bond* count also larger after `Chem.AddHs`?


In [ ]:
caffeine = good_mols[good_names.index("caffeine")]
print("heavy atoms:", caffeine.GetNumAtoms(), "bonds:", caffeine.GetNumBonds())

caffeine_h = Chem.AddHs(_)
print("all atoms:", caffeine_h._, "bonds:", caffeine_h._)
print("canonical SMILES:", Chem.MolToSmiles(caffeine))

### Displaying Multiple Molecules
The code below draws all ten surviving molecules in a grid with the compound names as legends.

In [ ]:
grid = Draw.MolsToGridImage(good_mols, molsPerRow=3, subImgSize=(220, 180), legends=good_names)
grid

#### Questions
12. Look at the three xanthines — caffeine, theophylline and theobromine. What differs between them structurally, and does the picture agree with what you know of their pharmacology?


## Part 2 - Atom counts as Arrays
The `Mol` object will be central to FA516/KE561.
It knows many properties for a molecule, like for example its size, that we saw above.

If we want to perform numeric operations (sums, averages, etc.) on numbers, it can be advantageous to use specialized lists (called arrays) from the package  [NumPy](https://numpy.org/doc/stable/).

### Predicting and Computing Molecular Weight
A rough but genuinely useful estimate of molecular weight from heavy-atom count alone is

$$\mathrm{MW} \approx 13.0 \times N_{\text{heavy}}$$

because the average heavy atom in a drug molecule is a carbon carrying about one hydrogen.
It is crude, and part of the task is finding out how crude.

But first we must import the NumPy library. Notice here, that we call it `np` for short - be aware of that in the exercises below.


In [ ]:
import numpy as np

#### Questions

1. Build a list `heavy_counts` of the heavy-atom count of every molecule in `good_mols`, using a [list comprehension](https://docs.python.org/3.10/tutorial/datastructures.html?highlight=list%20comprehension#list-comprehensions), and turn it into a NumPy array called `n_heavy`.
3. Compute `mw_estimate` from the formula above using that numpy does not require a loop to multiply with each element.


In [ ]:
heavy_counts = [mol.GetNumAtoms() for mol in good_mols]
n_heavy = np.array(_)
print(n_heavy)

mw_estimate = _
print(_)

The estimates we have now in `mw_estimate` are fine, but they are not worth anything unless we compare with the actual values of the molecules.

Here are the real molecular weights of the ten compounds, in g/mol, in the same order as `good_names`.

| name           | weight [g/mol] |
| -------------- | ------ |
| aspirin        | 180.16 |
| ibuprofen      | 206.28 |
| naproxen       | 230.26 |
| salicylic acid | 138.12 |
| celecoxib      | 381.37 |
| caffeine       | 194.19 |
| theophylline   | 180.16 |
| theobromine    | 180.17 |
| nicotine       | 162.23 |
| paracetamol    | 151.16 |

#### Questions
3. Make a new numpy array `mw_true` by transferring the molecular masses from the table above in the first line of code.
4. Compute `error` as the estimate from above minus the true molecular weight. Discuss what the output is.
5. Print the mean error and the standard deviation of the error using the two functions (`np.mean` and `np.std`).
6. Make the print more pretty by using the `round` function. *hint*: we used this function in the previous exercise.
7. The largest absolute error can be found with `np.max(np.abs(error))` - discuss what the code does and how it should be understood. Find which compound it belongs to by looking at the printed error array.
8. Is a mean error near zero enough to call this estimate good? Argue it with your neighbour (or yourself).

In [ ]:
mw_true = np.array([_])
error = _ - _
print(error)

## Part 3 - The Molecule Library as a DataFrame

*This is a natural place for the break — Part 2 is a finished result on its own.*

We have worked with four parallel lists: Names, classes, atom counts and molecular weights.
Keeping them aligned by hand is how mistakes happen so we will nuse the [DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) from the [pandas](https://pandas.pydata.org/docs/reference/index.html) library.

In last weeks exercise, we used `dict` for storage, but a DataFrame is the better choice.

We first import the pandas library and can refer to it as `pd`.


In [ ]:
import pandas as pd

### Data for DataScience
We also bring in two measured properties: the logarithm of the octanol/water partition coefficient, [logP](https://en.wikipedia.org/wiki/Partition_coefficient), and the number of [hydrogen bond donors](https://en.wikipedia.org/wiki/Hydrogen_bond).
With molecular weight and logP in the same table we can apply a real screening filter.

In the code below, the data is written to a [`.csv` file](https://en.wikipedia.org/wiki/Comma-separated_values) - a format that you have maybe encountered before when obtaining data from experiments.
The format is quite old, and not very robus, but you can easily read the format with your own eyes.

After the data is written to file (`fh.write`) we can load it using a pandas dataframe that natively reads the `.csv` file format.
Notice that we store the entire dataframe in the variable `mol_props`.

In [ ]:
csv_text = """name,logp,hbd
aspirin,1.19,1
ibuprofen,3.97,1
naproxen,3.18,1
salicylic acid,2.26,2
celecoxib,3.90,1
caffeine,-0.07,0
theophylline,-0.02,1
theobromine,-0.78,1
nicotine,1.17,0
paracetamol,0.46,2
"""
with open("library_props.csv", "w") as fh:
    fh.write(csv_text)

mol_props = pd.read_csv("library_props.csv")
mol_props

#### Questions
  1. Discuss with your neighbour what the LogP measure is. What does a negative number mean? what does a positive number mean?

It is possible to build a dataframe using a combination of lists and dictionaries.
Basically, a dictionary is used as input with the key acting as the column name and the list of data is the data in that column.

Luckily for us, we have the `good_` lists from above available to us.

### Questions

2. Build a DataFrame `df` from a dictionary with four columns: `name` from `good_names`, `class` from `good_classes`, `n_heavy` from `heavy_counts`, and `mw` from `mw_true`.
3. Add the `logp` and `hbd` columns by taking them from the `mol_props` dataframe we created above. The rows are already in the same order, so `mol_props["logp"]` can be used directly to assign a new column. *hint:* we did this in the first exercise last week.
4. Visualize the dataframe


In [ ]:
df = pd.DataFrame({"name": _, "class": _, "n_heavy": _, "mw": _})
df[_] = props["logp"]
df[_] = props["hbd"]
df

This is an impressive amount of data.
Our first task is to navigate this table using some of the ways to get the data we want.

For example, accessing entire rows of data is done using the [`iloc`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.iloc.html) functionality.
For example, the data of the entire first row is printed using the code below (notice it comes out as a column - don't worry about it):

In [ ]:
df.iloc[0]

#### Questions
  5. Why does the code below print the molecular mass of ibuprofen? Use the first table and the indices to see if you can spot why.

In [ ]:
df.iloc[1, 3]

Now we want to display only molecules we are interested in.
For this particular case, we will build two filters: one for molecular weight below 200 and one for logp above 1.0.
### Questions
6. Complete the filters in the code below.
7. use the `combined_filter` to get only the small lipophilic molecules.
8. Print only the `name`, `mw` and `logp` columns of that selection.
9. Which compounds passed, and does the combination "small and lipophilic" describe them fairly?


In [ ]:
filter_mw = df["mw"] < _
filter_logp = df["logp"] > _
filter_combined = filter_mw & filter_logp
small_lipophilic = df[_]
print(small_lipophilic[["name", "mw", "logp"]])

A real screening cascade does not filter on two properties chosen at random; it uses an agreed rule.
[Lipinski's rule of five](https://en.wikipedia.org/wiki/Lipinski%27s_rule_of_five) says an orally active drug usually has molecular weight below 500, logP below 5, no more than 5 hydrogen bond donors and no more than 10 acceptors.
We have three of the four descriptors, so we can apply three of the four conditions.

Once the subset exists, `describe()` gives the count, mean, spread and quartiles of every numeric column at once, which is the fastest way to characterise what survived a filter.


### Questions

10. Build `druglike_molecules` by keeping the rows with `mw` below 500, `logp` below 5 and `hbd` at most 5, and print how many compounds pass.
11. Print `describe()` for the `mw` and `logp` columns of `druglike`.
12. Every compound in this library passes. Is that a sign the filter works, or a sign the library was badly chosen for testing a filter?


In [ ]:
filter_mw = _
filter_logp = _
filter_hbd = _
filter_combined = filter_mw & filter_logp & filter_hbd
druglike = df[filter_combined]
print("number of passing compounds:", druglike.shape[0])
print(druglike[["mw", "logp"]].describe())

## Part 4 - Optional: repairing compound-X and annotating aspirin

You may stop here or finish this at home; nothing above depends on it.

Two loose ends remain.
The first is `compound-X`, whose SMILES `Cc1ccc(cc` never closed its ring or its branch.
The supplier confirms by e-mail that the intended structure is 4-methylbenzoic acid, SMILES `Cc1ccc(cc1)C(=O)O`, which is the same string with the ring digit and the bracket restored.

The second is annotation.
When you get atom indices back from a substructure search, you need to see which atoms those numbers mean, and `addAtomIndices` on the drawing options puts the numbers on the picture.


### Questions

1. Parse the corrected SMILES, confirm it is not `None`, and print its heavy-atom count and canonical SMILES.
2. What are the two differences between the broken string and the corrected one?
3. Its measured molecular weight is 136.15 g/mol and its logP is 2.27, with 1 hydrogen bond donor. Does it pass the three-part rule of five filter from Part 3?


In [ ]:
fixed = Chem.MolFromSmiles("Cc1ccc(cc1)C(=O)O")  #! the corrected SMILES from the supplier
print("parsed:", fixed is not None)
print("heavy atoms:", fixed.GetNumAtoms(), "canonical:", Chem.MolToSmiles(fixed))
print("passes filter:", 136.15 < 500 and 2.27 < 5 and 1 <= 5)  #! the three Lipinski conditions applied by hand
fixed

### Questions

4. Search aspirin for the carboxylic acid SMARTS pattern `C(=O)[OX2H1]` with `GetSubstructMatch` and print the matching atom indices.
5. Draw aspirin with `addAtomIndices` switched on and check the printed indices against the picture. Which atom of the two oxygens in the match is the acidic OH?


In [ ]:
pattern = Chem.MolFromSmarts("C(=O)[OX2H1]")
match = first_mol.GetSubstructMatch(pattern)  #! atom indices of the carboxylic acid in aspirin
print("carboxylic acid atoms:", match)
IPythonConsole.drawOptions.addAtomIndices = True
first_mol